In [1]:
using CSV
using Dates
using StringEncodings
using DataFrames

In [2]:
ENV["COLUMNS"] = 1000;

In [3]:
data_date = "2021_02_08";

In [4]:
hospitals = ["BMC", "HCGH", "JHH", "SH", "SMH"];

In [5]:
census_data = DataFrame(CSV.File(open(read, "../rawdata/realdata_$(data_date)/CensusWorksheet.csv", enc"UTF-16")));
active_data = filter(row -> !ismissing(row.CalcInstitutionName) && row.CalcInstitutionName != "JOHNS HOPKINS ALL CHILDREN'S HOSPITAL", census_data)
active_data = select(active_data,
    :CalcInstitutionName => (xs -> [string(split.(x, " ")[1]) for x in xs]) => :hospital,
    "Day of adate" => (x -> Date.(x, dateformat"U d, Y")) => :date,
    :CensusCount => :active,
);

In [6]:
function load_admissions(hname)
    adm = DataFrame(CSV.File(open(read, "../rawdata/realdata_$(data_date)/$(hname)CumAdmits.csv", enc"UTF-16")))
    adm = stack(adm, r"adate*")
    
    z = maximum([x for x in names(adm) if contains(x, "Column")])
    adm[:,z] = [ismissing(x) ? "date" : x for x in adm[:,z]]
    
    adm = unstack(adm, :variable, z, :value)
    select!(adm,
        :date,
        "Admissions to JHMI (excludes transfers)" => :admissions,
    )
    adm.date = [Date(d, dateformat"m/d/yyyy") for d in adm.date]
    adm.admissions = [parse(Float64, x) for x in adm.admissions]
    adm.admissions = [Int(x) for x in adm.admissions]
    
    return adm
end;

function load_admissions()
    data = []
    for hname in hospitals
        df = load_admissions(hname)
        insertcols!(df, 1, :hospital => fill(hname, nrow(df)))
        push!(data, df)
    end
    df = vcat(data...)
    return df
end;

In [7]:
admissions_data = load_admissions();

In [8]:
combined_data = outerjoin(active_data, admissions_data, on=[:hospital, :date])
combined_data.admissions = coalesce.(combined_data.admissions, 0)
sort!(combined_data, [:hospital, :date]);

In [9]:
combined_data |> CSV.write("../data/jhhs_realdata_$(data_date).csv");

In [10]:
combined_data

,hospital,date,active,admissions
,String?,Date?,Int64?,Int64
1,BMC,2020-03-12,0,0
2,BMC,2020-03-13,0,0
3,BMC,2020-03-14,0,0
4,BMC,2020-03-15,0,0
5,BMC,2020-03-16,0,0
6,BMC,2020-03-17,0,0
7,BMC,2020-03-18,1,2
8,BMC,2020-03-19,1,0
9,BMC,2020-03-20,0,0
